# Time-MoE

In [ ]:
!pip install matplotlib

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [ ]:
!pip install pyyaml
!pip install numpy
!pip install pandas
!pip install scikit-learn

In [ ]:
!pip install transformers==4.40.1

In [ ]:
#!pip install datasets==2.18.0

In [ ]:
!pip install accelerate==0.28.0

In [ ]:
!pip install accelerate

## Imports

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from transformers import AutoModelForCausalLM

import joblib

from sklearn.metrics import r2_score

In [ ]:
def timemoe_forecast(
    df,
    target_column,
    context_length,
    prediction_length,
    test_size,
    model_size='50M',
    device = 'cpu'
):
    data = torch.tensor(df[target_column].values, dtype=torch.float32).to(device)
    
    model = AutoModelForCausalLM.from_pretrained(
        f'Maple728/TimeMoE-{model_size}',
        device_map=device,
        trust_remote_code=True
    )
    
    all_predictions = []
    
    with torch.no_grad():
        for i in range(0, test_size - prediction_length + 1, prediction_length):
            # Get sequence for current window
            start_idx = len(data) - test_size + i - context_length
            sequence = data[start_idx:start_idx + context_length]
            sequence = sequence.unsqueeze(0)  # Add batch dimension
            
            # Normalize sequence
            mean = sequence.mean(dim=-1, keepdim=True)
            std = sequence.std(dim=-1, keepdim=True)
            normalized_sequence = (sequence - mean) / std
            
            # Generate forecast
            output = model.generate(
                normalized_sequence, 
                max_new_tokens=prediction_length
            )
            
            # Denormalize predictions
            normed_preds = output[:, -prediction_length:]
            predictions = normed_preds * std + mean
            all_predictions.append(predictions.squeeze(0).cpu())
    
    return torch.cat(all_predictions).numpy()

In [ ]:
# Read the dataset
aquifer_by_stations = joblib.load('aquifer_by_stations.joblib')
aquifers_list = [85065]

In [ ]:
horizon = 5 # prediction horizon
day_len = 200 # number of days to forecast

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(5)]

    # Iterate from day_len days before the end, to the last day
    for i in range(day_len + (horizon-1), 0, -1):
        y = aquifer_by_stations[aquifer]

        forecast = timemoe_forecast(
            df=y,
            target_column='altitude_diff',
            context_length=6*horizon,
            prediction_length=horizon,
            test_size=i,
            device='cuda'
        )

        # Store the results for every prediction horizon separately
        for i in range(horizon):
            #print(forecast.head())
            predictions[i].append(forecast[i])
    
    # Clean up the results
    predictions[0] = predictions[0][-200:]
    predictions[1] = predictions[1][3:-1]
    predictions[2] = predictions[2][2:-2]
    predictions[3] = predictions[3][1:-3]
    predictions[4] = predictions[4][0:-4]

    # Calculate the r2 scores and store them in a list
    for i in range(horizon):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

## Try number 2

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'Maple728/TimeMoE-200M',
    device_map="cpu",  # use "cpu" for CPU inference, and "cuda" for GPU inference.
    trust_remote_code=True,
)

In [ ]:
# Read the dataset
aquifer_by_stations = joblib.load('aquifer_by_stations.joblib')
aquifers_list = [85065]

In [ ]:
horizon = 5 # prediction horizon
day_len = 365 # number of days to forecast
context_length = 730

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(5)]

    with torch.no_grad():
        # Iterate from day_len days before the end, to the last day
        for j in range(day_len + (horizon-1), 0, -1):
            y = aquifer_by_stations[aquifer][-(j + context_length):-j]

            # Normalize the data
            mean, std = y['altitude_diff'].mean(), y['altitude_diff'].std()
            y['altitude_diff'] = (y['altitude_diff'] - mean) / std
            
            # Convert to tensor and add batch dimension, ensuring float32 dtype
            input_data = torch.tensor(y['altitude_diff'].values, dtype=torch.float32).unsqueeze(0)
            
            forecast = model.generate(
                inputs=input_data,
                max_new_tokens=horizon
            )
            
            # Convert back to numpy array
            forecast = forecast[0][-horizon:].cpu().numpy()
            forecast = forecast * std + mean

            # Store the results for every prediction horizon separately
            for i in range(horizon):
                predictions[i].append(forecast[i])
    
    # Clean up the results
    predictions[0] = predictions[0][-day_len:]
    predictions[1] = predictions[1][3:-1]
    predictions[2] = predictions[2][2:-2]
    predictions[3] = predictions[3][1:-3]
    predictions[4] = predictions[4][0:-4]

    # Calculate the r2 scores and store them in a list
    for i in range(horizon):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

In [ ]:
# Calculate the average r2 score
r2_average =  []
std_dev = []

for i in range(horizon):
    r2_average.append(np.mean(r2_scores[i]))
    std_dev.append(np.std(r2_scores[i]))

In [ ]:
r2_average

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], color="royalblue", label="true data")
plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[0], color="tomato", label="forecast0")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[1], color="orange", label="forecast1")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[2], color="green", label="forecast2")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[3], color="purple", label="forecast3")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[4], color="brown", label="forecast4")
plt.legend()
plt.grid()
plt.show()